# Detection Chain Calculations

This section will cover the calculations to determine how many photons are actually detected by the camera. As determined from [photon-budget.ipynb], 1.26E5 photons are produced per second. However, because of the geometry of the setup, only a certain amount of those photons will actually be usable for this project. Only a certain number of photons will reach the lens, then only a certain number will survive the optical path, and finally the quantum efficiency of the camera dictates how many of those that actuatlly reached it will be read. These initial calcualtions will be for a broadband estimate (400-550 nm), to understand if the project is feasible. The first step of this process is to find how many photons reach the lens. To do this, we will use a simple estimate:

$collection \approx \frac{1}{16N^2}$

In this equation, N represents the focal length of the lens. This equation is an order-of-magnitude estimate, assuming calibrated distance. The next steps are simple multiplications. Taking an estimate, 75% of the photons make it through the transmission path (0.95 (water) * 0.92 (glass) * 0.90 (lens)). This value will be measured on the physical device later on. The quantum effiency, finally, at 400-550 nm is ~0.83 for the camera chosen.

In [4]:
#code to calculate the amount of photos that reach the camera
transmission_rate = 0.95 * 0.92 * 0.90
def compute_photoelectrons_reaching_camera(lens, transmission_rate, quantum_efficiency, total_photons):
    return 1/(16*lens**2) * transmission_rate * quantum_efficiency * total_photons
print("Photoelectrons reaching camera: ", compute_photoelectrons_reaching_camera(1.4, transmission_rate, 0.83, 1.26e5))

Photoelectrons reaching camera:  2623.170535714286


Having established the rate of detected photoelectrons, the next question is whether this signal is distinguishable from noise, and how long an exposure is needed. This requires modeling the noise sources and computing the signal-to-noise ratio (SNR) as a function of integration time. In this project, there are 3 main sources of noise:
1. Shot noise
    - This is the fundamental noise that exists in photons going to the camera due to low light imaging. It is caused by an imprecision in measuring the discrete, random particles that will come into the camera.
2. Dark current
    - This noise exists due to thermal electrons existing when areas have heat. Because the camera will be cooled to roughly ~$-10\degree$ celsius, the dark current ends up being ~0.0006 $e^-$/px/s. (According to camera specification data)
3. Read Noise 
    - This noise is added everytime a sensor is read out. It is the fundamental noise in all of the electronics of the camera. It ends up as ~1.0 $e^-$/px per frame on high gain mode, and when stacking N frames scales as $\sqrt{N}$

As seen above, dark current is negligible at cooled temperatures (~$-10\degree$ C), so read noise and shot noise are the dominating factors. 
The signal to noise formula (SNR) per pixel can be written as:

$\frac{Signal}{\sqrt{signal + dark + read^2}}$

This signal is distributed over the pixels onto which the emitting region (the source + 1 cm beta halo) is imaged. A more concentrated image gives higher signal per pixel and faster detection. The exact pixel spread will be determined later in the real geometry. The detection threshold of SNR is when it reaches 5, and the quantititave measurement threshold is when it reaches 10. To find the required integration times, SNR is computed as a function of exposure time t. The signal accumulates linearly with time, while read noise accumulates per frame (√N over N frames) and shot noise grows as the square root of the accumulated signal. Solving for the time at which SNR reaches 5 (detection) and 10 (quantitative measurement) gives the required exposures.

In [3]:
import math
signal_rate = 2900        # detected e-/s total (from detection chain)
dark_current = 0.0006     # e-/px/s at -10C
read_noise = 1.0          # e-/px per frame
frame_time = 60           # s per sub-frame (typical for stacking)
def snr(t, pixels):
    signal = signal_rate * t
    signal_per_pixel = signal / pixels
    dark = dark_current * t
    read = math.sqrt(t/frame_time) * read_noise
    return signal_per_pixel / math.sqrt(signal_per_pixel + dark + read**2)
#SNR will be computed at a variety of pixel levels
#integration function
def time_to_reach_snr(target, pixels):
    t = 1
    while snr(t, pixels) < target:
        t += 1        
    return t
print("Time to reach SNR of 10 for 100 pixels: ", time_to_reach_snr(10, 100), "s")
print("Time to reach SNR of 10 for 1000 pixels: ", time_to_reach_snr(10, 1000), "s")
print("Time to reach SNR of 10 for 10000 pixels: ", time_to_reach_snr(10, 10000), "s")
print("Time to reach SNR of 10 for 100000 pixels: ", time_to_reach_snr(10, 100000), "s")

Time to reach SNR of 10 for 100 pixels:  4 s
Time to reach SNR of 10 for 1000 pixels:  35 s
Time to reach SNR of 10 for 10000 pixels:  366 s
Time to reach SNR of 10 for 100000 pixels:  5502 s


As seen above, the experiment closes in feasible exposure times. However, it scales very quickly with more pixels so it is important to priortize a small area when creating the physical geometry.